In [3]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)

print("="*50)
print("TRAINING LSTM MODEL")
print("="*50)

# ============================================
# 1. LOAD DATA
# ============================================
df = pd.read_csv(r"C:\Rainproj\rain.csv")

# ============================================
# 2. PREPROCESSING
# ============================================

# Date → Month
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df.drop('Date', axis=1, inplace=True)

# Drop missing target
df.dropna(subset=['RainTomorrow'], inplace=True)

# Encode target
df['RainTomorrow'] = df['RainTomorrow'].map({'Yes': 1, 'No': 0})

# Encode categorical
categorical_cols = df.select_dtypes(include=['object']).columns
label_encoders = {}

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown')
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Split X, y
X = df.drop('RainTomorrow', axis=1)
y = df['RainTomorrow']

# Handle missing values
num_cols = X.select_dtypes(include=[np.number]).columns
imputer = SimpleImputer(strategy='median')
X[num_cols] = imputer.fit_transform(X[num_cols])

# ============================================
# 3. TRAIN-TEST SPLIT
# ============================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ============================================
# 4. SCALING
# ============================================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ============================================
# 5. RESHAPE FOR LSTM
# ============================================
# LSTM expects 3D: (samples, timesteps, features)

X_train = X_train.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test = X_test.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("LSTM input shape:", X_train.shape)

# ============================================
# 6. BUILD LSTM MODEL
# ============================================
model = Sequential()

model.add(LSTM(64, return_sequences=False, input_shape=(1, X_train.shape[2])))
model.add(Dropout(0.3))

model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ============================================
# 7. TRAIN MODEL
# ============================================
early_stop = EarlyStopping(patience=5, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

# ============================================
# 8. PREDICTION
# ============================================
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# ============================================
# 9. EVALUATION
# ============================================
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("\n" + "="*40)
print("LSTM MODEL PERFORMANCE")
print("="*40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

# ============================================
# 10. SAVE MODEL
# ============================================
model.save("lstm_rain_model.h5")

print("\n✅ LSTM model saved!")

TRAINING LSTM MODEL
LSTM input shape: (113754, 1, 22)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 64)                  │          22,272 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 24,385 (95.25 KB)

 Trainable params: 24,385 (95.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 36s 9ms/step - accuracy: 0.8400 - loss: 0.3681 - val_accuracy: 0.8531 - val_loss: 0.3394
Epoch 2/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 29s 9ms/step - accuracy: 0.8463 - loss: 0.3527 - val_accuracy: 0.8569 - val_loss: 0.3389
Epoch 3/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 23s 7ms/step - accuracy: 0.8474 - loss: 0.3496 - val_accuracy: 0.8558 - val_loss: 0.3354
Epoch 4/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 16s 5ms/step - accuracy: 0.8499 - loss: 0.3467 - val_accuracy: 0.8550 - val_loss: 0.3331
Epoch 5/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8500 - loss: 0.3445 - val_accuracy: 0.8566 - val_loss: 0.3326
Epoch 6/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.8504 - loss: 0.3429 - val_accuracy: 0.8574 - val_loss: 0.3315
Epoch 7/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8516 - loss: 0.3420 - val_accuracy: 0.8558 - val_loss: 0.3303
Epoch 8/20
3200/3200 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8510 - loss: 0.3


LSTM MODEL PERFORMANCE
Accuracy:  0.8575
Precision: 0.7616
Recall:    0.5302
F1 Score:  0.6252

Confusion Matrix:
[[21006  1058]
 [ 2995  3380]]

✅ LSTM model saved!
